In [1]:
# capstone project, week 1

import numpy as np
import warnings
import matplotlib.pyplot as plt

from bayes_tools import (normalize, initial_bounds, generate_next_point, ucb_acquisition)
from viz_tools import (plot_1d_bo, plot_2d_bo)

# Function 1

In [2]:
X = np.load("initial_data/function_1/initial_inputs.npy")
y = np.load("initial_data/function_1/initial_outputs.npy")

In [3]:
n_initial = len(y)

bounds = initial_bounds(X, pad_fraction=1.0, lower_limit=0.0)

X, y = append_observations(X, y, new_X=x_next.reshape(1, -1), new_y=[the_result_you_got])


x_next, gp = generate_next_point(X, y, bounds, acquisition="pi", xi=0.01, maximize=True)
print("Evaluate this point:", x_next)  

# Week 2 -- append what you learned about LAST week's x_next, THEN propose again.
X, y = append_observations(X, y, new_X=x_next.reshape(1, -1), new_y=[the_result_you_got])
x_next, gp = generate_next_point(X, y, bounds, acquisition="pi", xi=0.01, maximize=True)
print("Evaluate this point:", x_next)

# Whenever you want the dashboard, using everything appended so far:
history = compute_iteration_diagnostics(X, y, bounds, n_initial=n_initial, acquisition="pi", xi=0.01, maximize=True)
fig = plot_bo_diagnostics(history, bounds)

SyntaxError: positional argument follows keyword argument (2344710475.py, line 5)

In [ ]:
# Get some info about what we loaded

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X range per dimension:")
print("  min:", X.min(axis=0))
print("  max:", X.max(axis=0))
print("y range:", y.min(), "to", y.max())

bounds = initial_bounds(X, pad_fraction=1.0)
print("\nBounds: ", bounds)



In [ ]:
x_next, gp = generate_next_point(
    X, y, bounds,
    acquisition="ucb",
    kappa=5.0,          # exploration-heavy
    maximize=True,
    n_restarts=25,
    random_state=0,
)

print("\n--- Next point to evaluate (bounds = [0,1]^3) ---")
print("x_next:", np.round(x_next, 4))

mu, sigma = gp.predict(normalize(x_next.reshape(1, -1), bounds), return_std=True)
print(f"GP predicted mean: {mu[0]:.4f}, predicted std: {sigma[0]:.4f}")

In [ ]:
fig = plot_2d_bo(
    X, y, bounds, gp,
    acquisition_fn=ucb_acquisition,
    x_next=x_next,
    acq_kwargs={"kappa": 5.0, "maximize": True},
)
plt.show()

# Iteration history tracking (diagnostics across weeks)

Since new `y` values arrive externally (submit `x_next`, find out the result later), the pattern below persists a small history log per function on disk. Each week:

1. If it's the very first time for this function, `init_history` from the initial batch.
2. If there's a pending proposal from last time, `record_observation` once you know its `y`.
3. `propose_and_log` to get the next `x_next` (this also records the acquisition value, GP hyperparameters, and domain-wide uncertainty at proposal time, for the diagnostic plots).
4. Plot diagnostics with the accumulated history -- no true optimum required.


In [ ]:
from bayes_tools import init_history, load_history, propose_and_log, record_observation
from viz_tools import plot_bo_diagnostics, plot_convergence, plot_acquisition_decay, \
                      plot_uncertainty_shrinkage, plot_sample_trajectory, plot_step_distance
import os


In [ ]:
# --- Function 1 -----------------------------------------------------------
history_path = "initial_data/function_1/history.npz"

X = np.load("initial_data/function_1/initial_inputs.npy")
y = np.load("initial_data/function_1/initial_outputs.npy")
bounds = initial_bounds(X, pad_fraction=1.0)

# Run once per function, ever -- this seeds the log with the initial batch.
if not os.path.exists(history_path):
    init_history(X, y, history_path)

history = load_history(history_path)
pending = np.isnan(history["y"])

if pending.any():
    print(f"Pending proposal awaiting a result: {history['X'][pending][0]}")
    print("Fill this in below once you know the true y, then re-run this cell.")
    # record_observation(history_path, y_observed=<value you got back>, x_observed=history['X'][pending][0])
else:
    x_next, gp = propose_and_log(
        history_path, bounds,
        acquisition="ucb", kappa=5.0, maximize=True,
        n_restarts=25, random_state=0,
        gp_kwargs={"n_restarts_optimizer": 20},
    )
    print("Next point to evaluate:", np.round(x_next, 4))


In [ ]:
# Once you know the result for the point above, record it, e.g.:
# record_observation(history_path, y_observed=0.42, x_observed=x_next)

history = load_history(history_path)

if (~np.isnan(history["y"])).sum() >= 2:  # need at least 2 proposals for a meaningful trend
    fig = plot_bo_diagnostics(history, bounds)
    plt.show()
else:
    print("Not enough completed iterations yet for the diagnostic dashboard "
          "(need at least one proposal with its y recorded).")
